In [1]:
import json
import collections
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
RESULTS_DIR = Path("../offline_bench/results")   # notebooks/ -> offline_bench/results/
X86_JSON = RESULTS_DIR / "bench_results.json"
BF3_JSON = RESULTS_DIR / "bench_results_bf3.json"

MODELS = ["16_8", "32_16", "64_32", "128_64_16", "256_128_32"]
MODEL_LABELS = ["(16,8)", "(32,16)", "(64,32)", "(128,64,16)", "(256,128,32)"]

# which (platform, kernel) series to plot, and how to label/color them.
# edit this if you want to add onnx / onnx-c into the same figure.
SERIES_SPEC = [
    ("bf3-arm", "scalar", "BF3 ARM scalar", "#a6bddb"),
    ("bf3-arm", "neon",   "BF3 ARM NEON",   "#045a8d"),
    ("cpu-x86", "scalar", "x86 scalar",     "#fdbb84"),
    ("cpu-x86", "avx",    "x86 AVX2/FMA",   "#b30000"),
]

# %%
def load_json_lines(*paths):
    """Read one-JSON-object-per-line result files into a flat list of dicts.
    Missing files are skipped with a warning rather than raising, so you can
    plot with only the x86 file present before the BF3 run is done."""
    records = []
    for p in paths:
        p = Path(p)
        if not p.exists():
            print(f"[WARN] missing results file: {p} -- skipping")
            continue
        with open(p) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                records.append(json.loads(line))
    return records


def to_series_dict(records, series_spec, models):
    """records -> {legend_label: [avg_latency_ns per model, in `models` order]}
    Uses the LAST matching record per (platform, kernel, model) so re-running
    a benchmark and appending to the same json just naturally supersedes the
    older number for that combo."""
    latest = {}
    for r in records:
        key = (r["platform"], r["kernel"], r["model"])
        latest[key] = r["avg_latency_ns"]  # last one wins (dict overwrite)

    series = collections.OrderedDict()
    for platform, kernel, label, color in series_spec:
        vals = []
        for m in models:
            v = latest.get((platform, kernel, m))
            if v is None:
                print(f"[WARN] no data for platform={platform} kernel={kernel} model={m}")
            vals.append(v)
        series[label] = vals
    return series

In [3]:
records = load_json_lines(X86_JSON, BF3_JSON)
print(f"loaded {len(records)} result records")

DATA = to_series_dict(records, SERIES_SPEC, MODELS)
COLORS = {label: color for _, _, label, color in SERIES_SPEC}
for label, vals in DATA.items():
    print(f"{label:16s} {vals}")

# %%
# ACM-ish publication style
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 7.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.6,
    "pdf.fonttype": 42,   # embed real fonts, not paths -- ACM/IEEE requirement
    "ps.fonttype": 42,
})

[WARN] missing results file: ..\results\bench_results.json -- skipping
[WARN] missing results file: ..\results\bench_results_bf3.json -- skipping
loaded 0 result records
[WARN] no data for platform=bf3-arm kernel=scalar model=16_8
[WARN] no data for platform=bf3-arm kernel=scalar model=32_16
[WARN] no data for platform=bf3-arm kernel=scalar model=64_32
[WARN] no data for platform=bf3-arm kernel=scalar model=128_64_16
[WARN] no data for platform=bf3-arm kernel=scalar model=256_128_32
[WARN] no data for platform=bf3-arm kernel=neon model=16_8
[WARN] no data for platform=bf3-arm kernel=neon model=32_16
[WARN] no data for platform=bf3-arm kernel=neon model=64_32
[WARN] no data for platform=bf3-arm kernel=neon model=128_64_16
[WARN] no data for platform=bf3-arm kernel=neon model=256_128_32
[WARN] no data for platform=cpu-x86 kernel=scalar model=16_8
[WARN] no data for platform=cpu-x86 kernel=scalar model=32_16
[WARN] no data for platform=cpu-x86 kernel=scalar model=64_32
[WARN] no data for 

In [4]:
def make_plot(data, colors, models, model_labels, out_prefix="mlp_bench_latency"):
    n_groups = len(models)
    n_series = len(data)
    x = np.arange(n_groups)
    width = 0.8 / n_series

    fig, ax = plt.subplots(figsize=(6.8, 3.0))  # ~2-column ACM width

    for i, (label, vals) in enumerate(data.items()):
        offset = (i - (n_series - 1) / 2) * width
        vals_plot = [v if v is not None else 0 for v in vals]
        ax.bar(x + offset, vals_plot, width, label=label, color=colors[label],
               edgecolor="black", linewidth=0.4)

    ax.set_yscale("log")
    ax.set_ylabel("Mean inference latency (ns, log scale)")
    ax.set_xlabel("MLP hidden layer sizes")
    ax.set_xticks(x)
    ax.set_xticklabels(model_labels)
    ax.yaxis.grid(True, which="major", linestyle="--", linewidth=0.4, alpha=0.6)
    ax.set_axisbelow(True)
    ax.legend(ncol=len(data), loc="upper center", bbox_to_anchor=(0.5, 1.22),
              frameon=False, columnspacing=1.2, handlelength=1.4)

    # speedup annotations: vector-over-scalar per platform, wherever both
    # a "*scalar*" and a non-scalar series exist for the same platform prefix
    labels = list(data.keys())
    scalar_labels = [l for l in labels if "scalar" in l.lower()]
    vector_labels = [l for l in labels if "scalar" not in l.lower()]
    for s_label in scalar_labels:
        platform_prefix = s_label.split(" scalar")[0].split(" Scalar")[0]
        for v_label in vector_labels:
            if v_label.startswith(platform_prefix):
                si = labels.index(s_label)
                vi = labels.index(v_label)
                s_off = (si - (n_series - 1) / 2) * width
                v_off = (vi - (n_series - 1) / 2) * width
                for i in range(n_groups):
                    sv, vv = data[s_label][i], data[v_label][i]
                    if sv is None or vv is None:
                        continue
                    speedup = sv / vv
                    ax.text(x[i] + v_off, vv * 1.15, f"{speedup:.1f}x",
                            ha="center", va="bottom", fontsize=6.5, color=colors[v_label])

    fig.tight_layout()
    fig.savefig(f"{out_prefix}.pdf", bbox_inches="tight")
    fig.savefig(f"{out_prefix}.png", dpi=300, bbox_inches="tight")
    print(f"wrote {out_prefix}.pdf and {out_prefix}.png")
    return fig

In [5]:
fig = make_plot(DATA, COLORS, MODELS, MODEL_LABELS)

C:\Users\Sergio\AppData\Local\Temp\ipykernel_35952\4015602480.py:15: UserWarning: Data has no positive values, and therefore cannot be log-scaled.
  ax.set_yscale("log")


wrote mlp_bench_latency.pdf and mlp_bench_latency.png
